Cell 1 — Imports + configuration

In [ ]:
# ============================================================
# QUERY GENERATION CONFIGURATION
# ============================================================

import os
import json
import random
import re
import time
import mimetypes
from pathlib import Path

import pandas as pd
from google.genai import types


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

# Change these three paths
TEMPLATE_CSV = "/kaggle/input/datasets/deboparna9das/csvs-queries/human_queries_template.csv"
TOOL_REFERENCE_CSV = "/kaggle/input/datasets/deboparna9das/csvs-queries/tool_reference.csv"

# Folder containing images
IMAGE_FOLDER = "/kaggle/input/datasets/deboparna9das/images-123"


# ------------------------------------------------------------
# PILOT CONFIG
# ------------------------------------------------------------

NUM_IMAGES = 50

# Generate up to 3 queries per image
QUERIES_PER_IMAGE = 3

# Maximum number of retries for an invalid candidate
MAX_RETRIES_PER_SLOT = 5

# Random seed for reproducibility
RANDOM_SEED = 42

random.seed(RANDOM_SEED)


# ------------------------------------------------------------
# RATE LIMITING (new)
# ------------------------------------------------------------

# Minimum seconds between ANY two Gemini calls.
# 6s = max ~10 calls/minute. Raise this if you still see errors,
# lower it if generation is running clean and you want more speed.
MIN_SECONDS_BETWEEN_CALLS = 6

# Base delay used for exponential backoff after an API error.
# attempt 1 -> 5s, attempt 2 -> 10s, attempt 3 -> 15s, ...
RETRY_BASE_DELAY = 5


# ------------------------------------------------------------
# FALLBACK TOOLS (new)
# ------------------------------------------------------------
# Used only when an image gets ZERO accepted queries through
# normal balanced tool/structure selection. Tried in this order.
# Each candidate is STILL checked for genuine applicability by
# Gemini -- it is never forced onto the image.

FALLBACK_TOOLS = [
    "object_detection",
    "background_removal",
    "salient_object_segmentation"
]


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_CSV = "/kaggle/working/human_queries_pilot_50images.csv"

print("Configuration loaded.")
print("Image folder:", IMAGE_FOLDER)
print("Template:", TEMPLATE_CSV)
print("Tool reference:", TOOL_REFERENCE_CSV)
print("Output:", OUTPUT_CSV)
print(f"Rate limit: 1 call every {MIN_SECONDS_BETWEEN_CALLS}s")


Cell 2 — Load template + tool reference

In [ ]:
# ============================================================
# LOAD CSV FILES
# ============================================================

template_df = pd.read_csv(TEMPLATE_CSV)
tool_ref_df = pd.read_csv(TOOL_REFERENCE_CSV)

print("Template shape:", template_df.shape)
print("Tool reference shape:", tool_ref_df.shape)

print("\nTemplate columns:")
print(template_df.columns.tolist())

print("\nTool reference columns:")
print(tool_ref_df.columns.tolist())

print("\nTool reference preview:")
display(tool_ref_df.head())

Cell 3 — Validate the tool reference

In [ ]:
# ============================================================
# VALIDATE TOOL REFERENCE
# ============================================================

required_tool_columns = {"tool_id", "tool_name"}

missing = required_tool_columns - set(tool_ref_df.columns)

if missing:
    raise ValueError(
        f"tool_reference.csv is missing columns: {missing}"
    )

# Remove accidental duplicate tool names
tool_ref_df = tool_ref_df.drop_duplicates(
    subset=["tool_name"]
).reset_index(drop=True)

# List of 41 tool names
TOOLS = tool_ref_df["tool_name"].astype(str).tolist()

# Map:
# tool_name -> tool_id
TOOL_ID_MAP = dict(
    zip(
        tool_ref_df["tool_name"].astype(str),
        tool_ref_df["tool_id"]
    )
)

print(f"Number of tools found: {len(TOOLS)}")

if len(TOOLS) != 41:
    print(
        f"WARNING: Expected 41 tools, but found {len(TOOLS)} "
        "in tool_reference.csv."
    )

print("\nTools:")
for name in TOOLS:
    print(f"{TOOL_ID_MAP[name]} -> {name}")

Cell 4 — Define the five query structures

In [ ]:
# ============================================================
# QUERY STRUCTURES
# ============================================================

QUERY_STRUCTURES = {

    "direct_tool_request": """
The user explicitly asks for a computer vision operation.
The wording may directly mention what should be detected,
segmented, classified, estimated, enhanced, etc.

Example:
"Detect all faces in this image."
""",

    "natural_user_phrasing": """
The user describes what they want naturally without using
technical computer vision terminology.

Example:
"Can you tell what is happening here?"
""",

    "ambiguous_phrasing": """
The request is intentionally vague and could reasonably
correspond to more than one computer vision capability.

Example:
"What is unusual in this image?"
""",

    "visually_grounded": """
The request asks something whose answer depends directly
on visual information in the provided image.

Example:
"How far away is the car?"
""",

    "overlapping_tool_case": """
The query lies near the boundary between two or more
computer vision tools.

Examples:
"Can you separate each person from the background?"
This may involve segmentation or matting.

"What is happening in this image?"
This may involve captioning, VQA, action recognition,
or visual reasoning.
"""
}

STRUCTURE_NAMES = list(QUERY_STRUCTURES.keys())

print(STRUCTURE_NAMES)

Cell 5 — Tool descriptions

In [ ]:
# ============================================================
# TOOL DESCRIPTIONS
# ============================================================

TOOL_DESCRIPTIONS = {

    "object_detection":
        "Locate and identify individual objects, typically using bounding boxes.",

    "object_counting":
        "Count visible objects of a specified category.",

    "person_detection":
        "Locate people in an image.",

    "face_detection":
        "Locate human faces in an image.",

    "vehicle_detection":
        "Locate vehicles such as cars, buses, trucks, motorcycles, bicycles, etc.",

    "animal_detection":
        "Locate and identify visible animals.",

    "text_detection_ocr":
        "Detect visible text and/or extract readable text from the image.",

    "logo_detection":
        "Locate and identify visible logos or brand marks.",

    "anomaly_detection":
        "Identify visually unusual, abnormal, damaged, defective, or unexpected elements.",

    "semantic_segmentation":
        "Assign a semantic class to every pixel without separating individual instances of the same class.",

    "instance_segmentation":
        "Separate individual object instances at the pixel level.",

    "panoptic_segmentation":
        "Perform pixel-level semantic segmentation while distinguishing individual object instances.",

    "background_removal":
        "Separate the foreground subject from the background.",

    "salient_object_segmentation":
        "Identify and segment the visually most prominent or salient object or region.",

    "matting":
        "Perform fine-grained foreground extraction with precise boundaries, including hair, fur, and semi-transparent regions.",

    "text_to_image_generation":
        "Generate a new image from a textual description.",

    "style_transfer":
        "Transform an image into a different visual or artistic style while preserving its content.",

    "super_resolution":
        "Increase image resolution while reconstructing plausible missing details.",

    "image_colorization":
        "Add plausible color to a grayscale or monochrome image.",

    "image_denoising":
        "Remove unwanted noise or grain from an image while preserving meaningful structure.",

    "image_deblurring":
        "Recover visual detail from blurred imagery.",

    "depth_estimation":
        "Estimate the depth or relative distance of visible regions or objects.",

    "surface_normal_estimation":
        "Estimate the 3D orientation of visible surfaces.",

    "3d_reconstruction":
        "Reconstruct a 3D representation of an object or scene.",

    "point_cloud_generation":
        "Represent a 3D scene or object as a collection of spatial points.",

    "image_classification":
        "Assign one or more broad categories to the entire image.",

    "fine_grained_classification":
        "Identify a highly specific subtype, species, model, or category.",

    "scene_recognition":
        "Identify the type or category of environment or scene.",

    "facial_expression_recognition":
        "Determine visible facial expressions represented by faces.",

    "age_gender_estimation":
        "Estimate apparent age and gender-related visual categories from visible people.",

    "action_recognition":
        "Identify actions or activities being performed.",

    "pose_estimation":
        "Estimate human body keypoints, posture, or pose.",

    "gaze_estimation":
        "Estimate the direction in which a person's eyes or gaze are directed.",

    "visual_question_answering":
        "Answer a natural-language question using information visible in the image.",

    "image_captioning":
        "Generate a natural-language description of the image.",

    "visual_reasoning":
        "Infer relationships, events, intent, or conclusions that require reasoning over multiple visual elements.",

    "visual_grounding":
        "Identify the specific visual region or object referred to by a natural-language expression.",

    "relation_detection":
        "Identify semantic relationships between visible objects.",

    "attribute_recognition":
        "Identify visual properties or attributes of objects.",

    "video_object_tracking":
        "Track an object across multiple video frames.",

    "temporal_action_localization":
        "Determine when an action occurs within a video."
}

Cell 6 — Image discovery

In [ ]:
# ============================================================
# DISCOVER IMAGES
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp"
}

image_paths = sorted(
    [
        p for p in Path(IMAGE_FOLDER).iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]
)

print(f"Total images found: {len(image_paths)}")

if len(image_paths) < NUM_IMAGES:
    raise ValueError(
        f"Only {len(image_paths)} images found, "
        f"but NUM_IMAGES={NUM_IMAGES}"
    )

# Randomly select exactly 20 for the pilot
selected_images = random.sample(
    image_paths,
    NUM_IMAGES
)

print("\nSelected images:")
for p in selected_images:
    print(p.name)

Cell 7 — Helper to extract image ID

In [ ]:
# ============================================================
# IMAGE ID
# ============================================================

def get_image_id(image_path):
    """
    Image filename is the prompt/image ID.

    Examples:
        397133.jpg       -> 397133
        abc123.png       -> abc123
        prompt_001.jpeg  -> prompt_001
    """
    return Path(image_path).stem

Cell 8 — Quota system

In [ ]:
# ============================================================
# TOOL + STRUCTURE BALANCING
# ============================================================

from PIL import Image
import numpy as np

tool_counts = {
    tool: 0
    for tool in TOOLS
}

structure_counts = {
    structure: 0
    for structure in STRUCTURE_NAMES
}

tool_structure_counts = {
    (tool, structure): 0
    for tool in TOOLS
    for structure in STRUCTURE_NAMES
}
tool_attempt_counts = {
    tool: 0
    for tool in TOOLS
}

# ------------------------------------------------------------
# Tools that structurally cannot apply to a SINGLE STILL IMAGE
# dataset (they require video/multi-frame input). Excluding
# them from selection stops them from hogging every retry.
# ------------------------------------------------------------

TOOLS_ALWAYS_EXCLUDED = {
    "video_object_tracking",
    "temporal_action_localization"
}


def is_grayscale_image(image_path, threshold=10):
    """
    Cheap local check (no API call) -- image_colorization only
    makes sense for genuinely grayscale/monochrome photos.
    """
    try:
        img = Image.open(image_path).convert("RGB").resize((64, 64))
        arr = np.asarray(img).astype(int)
        diff = (
            np.abs(arr[:, :, 0] - arr[:, :, 1])
            + np.abs(arr[:, :, 1] - arr[:, :, 2])
            + np.abs(arr[:, :, 0] - arr[:, :, 2])
        )
        return diff.mean() < threshold
    except Exception:
        return False


def get_selectable_tools(image_path):
    pool = [t for t in TOOLS if t not in TOOLS_ALWAYS_EXCLUDED]

    if "image_colorization" in pool and not is_grayscale_image(image_path):
        pool = [t for t in pool if t != "image_colorization"]

    return pool


def choose_target_tool(image_path):
    """
    Prefer tools that have been ATTEMPTED the fewest times so far
    (not just accepted). This stops a tool that's rarely applicable
    (e.g. vehicle_detection on an image with no vehicles) from
    permanently sitting at the bottom and hogging every retry slot,
    since its attempt count rises even when it gets rejected.
    """
    pool = get_selectable_tools(image_path)

    min_attempts = min(tool_attempt_counts[t] for t in pool)

    candidates = [
        tool
        for tool in pool
        if tool_attempt_counts[tool] == min_attempts
    ]

    return random.choice(candidates)


def choose_target_structure(tool):
    """
    Prefer query structures that have fewer examples for
    the selected tool.
    """
    counts = {
        structure: tool_structure_counts[(tool, structure)]
        for structure in STRUCTURE_NAMES
    }

    min_count = min(counts.values())

    candidates = [
        structure
        for structure, count in counts.items()
        if count == min_count
    ]

    return random.choice(candidates)

Cell 9 — Gemini prompt

In [ ]:
# ============================================================
# COMBINED GEMINI PROMPT FUNCTION
# ============================================================

def build_generation_prompt(tool_name, query_structure):

    # --------------------------------------------------------
    # Validate inputs
    # --------------------------------------------------------

    if tool_name not in TOOL_DESCRIPTIONS:
        raise ValueError(
            f"Unknown tool: {tool_name}"
        )

    if query_structure not in QUERY_STRUCTURES:
        raise ValueError(
            f"Unknown query structure: {query_structure}"
        )

    # --------------------------------------------------------
    # Selected tool information
    # --------------------------------------------------------

    tool_description = TOOL_DESCRIPTIONS[tool_name]

    structure_description = QUERY_STRUCTURES[
        query_structure
    ]

    # --------------------------------------------------------
    # Combined prompt
    # --------------------------------------------------------

    prompt = f"""
You are an expert computer vision dataset annotator.

You are given ONE IMAGE.

Your job is to inspect the actual image and determine whether a
TARGET COMPUTER VISION TOOL is genuinely applicable to that image.
If it is applicable, generate ONE natural user query corresponding
to that tool and to the requested query structure.

Do NOT force a tool onto an image simply because something vaguely
related to the tool appears.

============================================================
GLOBAL APPLICABILITY RULES
============================================================

A tool is APPLICABLE only when:

1. The visual evidence necessary for the task is actually present.
2. The requested operation genuinely matches the tool capability.
3. The query would naturally be something a real user could ask
   about this specific image.
4. The tool is not being selected merely because an object related
   to its name happens to appear.
5. A simpler or more appropriate tool should not obviously replace it,
   unless the query is intentionally an overlapping-tool case.
6. Do not invent objects, text, actions, relationships, defects,
   transparency, distance, gaze, expressions, scene properties,
   or other visual information.
7. Do not assume properties that cannot reasonably be established
   from the image.
8. If visual evidence is insufficient, mark the tool as NOT applicable.
9. A tool should be rejected when its use would feel artificial or
   forced for a competent computer-vision practitioner.

IMPORTANT PRINCIPLE:

Ask yourself:

"Would a competent computer-vision practitioner actually choose
THIS tool for THIS query on THIS image?"

If the answer is no, return applicable=false.

============================================================
IMPORTANT CROSS-TOOL DISTINCTIONS
============================================================

OBJECT DETECTION
= locate and identify objects.

OBJECT COUNTING
= determine how many objects are present.

IMAGE CLASSIFICATION
= assign a broad class/category to the whole image.

FINE-GRAINED CLASSIFICATION
= identify a highly specific subtype/species/model/breed/etc.

PERSON DETECTION
= locate whole people.

FACE DETECTION
= locate faces.

ACTION RECOGNITION
= identify a visible action/activity.

POSE ESTIMATION
= estimate body keypoints/posture.

GAZE ESTIMATION
= estimate where a person is looking.

FACIAL EXPRESSION RECOGNITION
= identify visible facial expression.

VISUAL QUESTION ANSWERING
= answer a natural-language question about visual content.

IMAGE CAPTIONING
= provide a general natural-language description.

VISUAL REASONING
= perform reasoning/inference over multiple visual elements.

VISUAL GROUNDING
= locate a specific object/region referred to by language.

RELATION DETECTION
= identify relationships between objects.

ATTRIBUTE RECOGNITION
= identify visible properties/attributes of objects.

SEMANTIC SEGMENTATION
= label pixels by semantic class without separating same-class instances.

INSTANCE SEGMENTATION
= provide separate masks for individual object instances.

PANOPTIC SEGMENTATION
= semantic pixel labeling + individual instance separation.

BACKGROUND REMOVAL
= ordinary foreground/background separation.

MATTING
= highly precise alpha-level foreground separation, especially
for difficult boundaries such as transparency, hair, fur,
smoke, fine strands, or semi-transparent regions.

DEPTH ESTIMATION
= estimate distance/depth.

SURFACE NORMAL ESTIMATION
= estimate orientation of surfaces.

3D RECONSTRUCTION
= reconstruct a structured 3D representation.

POINT CLOUD GENERATION
= create a 3D point-based representation.

IMAGE DENOISING
= remove noise/grain.

IMAGE DEBLURRING
= recover information lost through blur.

SUPER RESOLUTION
= increase image resolution.

IMAGE COLORIZATION
= add color to grayscale/monochrome imagery.

STYLE TRANSFER
= change artistic/visual style while preserving content.

ANOMALY DETECTION
= identify meaningful abnormalities, defects, damage, or unexpected
visual patterns.

SALIENCY SEGMENTATION
= segment the visually dominant/salient object or region.

OCR/TEXT DETECTION
= detect/read visible textual content.

LOGO DETECTION
= locate/identify recognizable logos or brand marks.

VIDEO OBJECT TRACKING
= track the same object through multiple video frames.

TEMPORAL ACTION LOCALIZATION
= determine when an action occurs within a video.

============================================================
TARGET TOOL
============================================================

Tool name:
{tool_name}

Tool definition:
{tool_description}

The TARGET TOOL is the tool that you must evaluate against
the actual image.

Do NOT automatically accept it.

============================================================
TARGET QUERY STRUCTURE
============================================================

Structure:
{query_structure}

Definition:
{structure_description}

The generated query MUST follow this structure.

The structure and ambiguity level are separate concepts.

============================================================
FIVE QUERY STRUCTURES
============================================================

1. direct_tool_request

The user explicitly asks for a computer-vision operation.

Examples:
"Detect all faces in this image."
"Remove the background."
"Segment each person."

2. natural_user_phrasing

The user asks naturally without technical computer-vision terminology.

Examples:
"Can you tell me what is happening here?"
"What do you see in this picture?"

3. ambiguous_phrasing

The wording is intentionally vague and could reasonably correspond
to multiple computer-vision capabilities.

Examples:
"What is unusual in this image?"
"What stands out here?"

4. visually_grounded

The query directly depends on visual information from this
particular image, such as an object's location, distance,
appearance, orientation, or spatial relationship.

Examples:
"How far away is the car?"
"Which glass is closer to the menu?"

5. overlapping_tool_case

The query genuinely sits near the boundary between two or more
tools.

Examples:
"Can you separate the glass from the table?"
could involve segmentation, background removal, or matting.

"What's happening in this picture?"
could involve captioning, action recognition, VQA, or reasoning.

For overlapping cases, the query should have a REAL overlap.
Do not artificially make a straightforward query ambiguous.

============================================================
AMBIGUITY LEVEL
============================================================

LOW:
The intended target tool is clearly the best match and competing
tools are unlikely.

MEDIUM:
One or two alternative tools could reasonably be considered.

HIGH:
Several tools could plausibly answer or perform the request,
or the wording is genuinely underspecified.

IMPORTANT:

Do not assign ambiguity based only on the requested query structure.

For example:

"Detect all faces in this image."
-> direct_tool_request
-> low ambiguity

"Separate each person from the background."
-> overlapping_tool_case
-> medium ambiguity

"What is unusual in this image?"
-> ambiguous_phrasing
-> potentially high ambiguity

============================================================
TOOL-SPECIFIC APPLICABILITY RULES
============================================================

OBJECT DETECTION:
Applicable when the image contains identifiable objects and the query
asks to detect, locate, or identify them.
Do not use when the primary task is counting, segmentation,
classification, or reasoning.
OBJECT DETECTION — IMPORTANT QUERY REQUIREMENT:

The query should involve identifying and/or locating specific visible
objects.

A query that only asks for a broad description of what exists may
overlap too strongly with image captioning or classification.

Prefer natural queries such as:
- "What objects can you identify on the table?"
- "Can you point out the different items on the table?"
- "Where are the glasses and pizza boxes?"

Avoid overly broad queries such as:
- "What is happening in this image?"
- "What is this image about?"

When the query only asks which objects are present without any
localization, ambiguity may be MEDIUM rather than LOW because
image captioning or other tools could also plausibly answer it.

OBJECT COUNTING:
Applicable when the query requires counting visible instances.
Do not use merely because multiple objects exist.

PERSON DETECTION:
Applicable when visible people must be located.
Do not use merely because a person's attributes, action, pose,
expression, or gaze are being analyzed.

FACE DETECTION:
Applicable when sufficiently visible human faces must be located.
Do not use if there are no visible faces or they are too unclear.

VEHICLE DETECTION:
Applicable when relevant vehicles are visibly present and need
to be detected/located.
Do not force it when no vehicle is visible.

ANIMAL DETECTION:
Applicable when an animal is actually visible.
Do not infer an animal from ambiguous shapes.

TEXT DETECTION / OCR:
Applicable when readable or locatable text is visibly present and
the query asks to detect/read/extract it.
Do not invent text.

LOGO DETECTION:
Applicable only when a recognizable logo/brand mark is actually visible.
A generic product does not automatically imply logo detection.

ANOMALY DETECTION:
Applicable when there is a meaningful abnormality, defect, damage,
or unexpected visual element.
Do not call something anomalous merely because it is unusual
without contextual justification.

SEMANTIC SEGMENTATION:
Applicable when the query requires pixel-level semantic classes.
Same-class individual objects do not need to be separated.

INSTANCE SEGMENTATION:
Applicable when individual object instances need precise pixel masks.
Do not use when bounding boxes are sufficient.

PANOPTIC SEGMENTATION:
Applicable when the whole scene needs pixel-level semantic labeling
while individual object instances are separately distinguished.

BACKGROUND REMOVAL:
Applicable for ordinary foreground/background isolation.
Do not assume every object-isolation request requires matting.

SALIENCY SEGMENTATION:
Applicable when the query specifically concerns the visually dominant
or salient object/region AND requires segmentation of it.

MATTING:
Applicable only when fine alpha-level boundaries matter.
Strong cases include:
- transparent objects
- translucent materials
- hair
- fur
- smoke
- fine strands
- semi-transparent regions
- difficult foreground boundaries

CRITICAL MATTING RULE:
The mere presence of a glass, transparent-looking object, hair,
or another potentially difficult material does NOT automatically
make matting applicable.

For example:
"Extract the glass from the table."
is not sufficient by itself.

The image must provide a genuine reason why fine-grained matting
is useful or necessary.

TEXT-TO-IMAGE GENERATION:
Applicable when the task asks to create a new image.
Since this dataset is image-grounded, preferably use it only when
the generated request meaningfully refers to the supplied image
as a reference/transformation source.

STYLE TRANSFER:
Applicable when the query asks to transform the artistic/visual
style of the supplied image while preserving its content.

SUPER RESOLUTION:
Applicable when the image is visibly low-resolution/pixelated or
the query explicitly asks for upscaling.
Do not use merely because the user wants a "better" image.

IMAGE COLORIZATION:
Applicable primarily to grayscale/monochrome images.
Reject for ordinary color photographs.

IMAGE DENOISING:
Applicable when visible random noise, grain, or sensor artifacts
are present or explicitly relevant.
Do not confuse noise with blur.

IMAGE DEBLURRING:
Applicable when visible blur is present, such as motion blur or
defocus blur.
Do not confuse blur with noise or low resolution.

DEPTH ESTIMATION:
Applicable when the query asks about depth, distance, or relative
foreground/background distance.
Do not treat ordinary 2D location as depth.

SURFACE NORMAL ESTIMATION:
Applicable when the query concerns the orientation/direction of
visible surfaces in 3D.

3D RECONSTRUCTION:
Applicable when the query asks to reconstruct/model the visible
object or scene in 3D.
Do not claim knowledge about completely unseen geometry.

POINT CLOUD GENERATION:
Applicable when the query explicitly asks for a 3D point-cloud
representation.

IMAGE CLASSIFICATION:
Applicable when the query asks for a broad category/class for
the overall image.
Do not use for object-level localization.

FINE-GRAINED CLASSIFICATION:
Applicable when the image contains enough evidence for a specific
subtype, species, breed, model, or other fine category.
Do not invent fine-grained identities from insufficient evidence.

SCENE RECOGNITION:
Applicable when the query concerns the type of environment,
location, or overall scene.
Do not use simply because the image has a background.

FACIAL EXPRESSION RECOGNITION:
Applicable only when a sufficiently visible face provides enough
evidence of expression.
Do not infer expression from body posture alone.

AGE/GENDER ESTIMATION:
Applicable when a person's visible appearance is sufficiently clear
to support an estimate.
Treat the result as an estimate rather than certain fact.

ACTION RECOGNITION:
Applicable when an action/activity is visibly supported.
Do not infer that somebody is performing an action merely because
the relevant object is nearby.

POSE ESTIMATION:
Applicable when a sufficiently visible human body allows posture
or body-keypoint analysis.
Presence of a person alone is insufficient.

GAZE ESTIMATION:
Applicable only when a person's eyes/face are sufficiently visible
to infer gaze direction.
Do not use body direction alone as gaze.

VISUAL QUESTION ANSWERING:
Applicable when the user is asking a question whose answer can be
obtained from the image.
For highly specialized tasks, prefer the specialized tool when
that tool is clearly intended.

IMAGE CAPTIONING:
Applicable when the user wants a natural-language description of
the image or what is happening in it.
Do not use if a specific question or specialized operation is clearly
the real task.

VISUAL REASONING:
Applicable when answering requires combining multiple visual clues,
inferring intent, likely events, spatial reasoning, or conclusions
beyond direct recognition.
Do not use for simple object detection or directly visible facts.

VISUAL GROUNDING:
Applicable when the query refers to a specific visual object or region
through language and that object/region needs to be located.

RELATION DETECTION:
Applicable when the query asks about the relationship between visible
objects, such as on, behind, next to, holding, riding, inside, etc.

ATTRIBUTE RECOGNITION:
Applicable when the query asks about a visible property of an object,
such as color, shape, material, pattern, or state.

VIDEO OBJECT TRACKING:
This requires multiple video frames.
If the provided input is a single still image, return applicable=false.

TEMPORAL ACTION LOCALIZATION:
This requires temporal/video information.

If the provided input is a single still image, return applicable=false.
============================================================
QUERY-TOOL CONSISTENCY CHECK
============================================================

Before returning applicable=true, verify ALL of the following:

1. If the target tool were removed from the taxonomy, would the
   query still clearly require that capability?

2. Could a substantially simpler/general tool answer the query
   without performing the target operation?

3. Does the wording of the query itself express the target task?

4. Do the objects listed support the query?

5. Is the query grounded in something actually visible?

6. Are the notes consistent with the actual query?

If the answer to these checks is NO, return applicable=false
rather than forcing the target tool.

IMPORTANT:
Do not use the notes to justify a mismatch between the query
and the target tool.

The QUERY itself must be a valid example of the target tool.

============================================================
FINAL DECISION PROCESS
============================================================

Before generating the answer, perform these steps:

STEP 1:
Inspect the actual image carefully.

STEP 2:
Identify the relevant visible objects, regions, text, people,
scene characteristics, actions, and other evidence.

STEP 3:
Ask whether the TARGET TOOL is genuinely appropriate.

STEP 4:
If not appropriate, return applicable=false.

STEP 5:
If appropriate, create ONE query following the TARGET QUERY STRUCTURE.

STEP 6:
Check that the query does not invent information.

STEP 7:
Check whether another simpler or more appropriate tool would
normally be chosen.

STEP 8:
Assign ambiguity_level based on actual tool overlap.

============================================================
OUTPUT FORMAT
============================================================

Return ONLY valid JSON.

If the target tool is NOT applicable:

{{
  "applicable": false,
  "query": null,
  "objects": [],
  "ambiguity_level": null,
  "requires_image_context": "yes",
  "notes": "One short sentence only (max 12 words) on why not applicable."
}}

If the target tool IS applicable:

{{
  "applicable": true,
  "query": "ONE natural user query",
  "objects": ["important", "visible", "objects"],
  "ambiguity_level": "low",
  "requires_image_context": "yes",
  "notes": "One short sentence only (max 12 words) on why this fits."
}}

============================================================
FINAL QUALITY REQUIREMENTS
============================================================
QUERY MUST TEST THE TARGET TOOL

The generated query must require the actual capability of the
TARGET TOOL to answer or perform.

Do NOT generate a broad query merely because the target tool could
potentially contribute to the answer.

For example:

Target tool = relation_detection

BAD:
"What is notable about the items on the table?"
This mainly asks for a general description and could be answered
through captioning, object detection, attribute recognition,
or VQA.

GOOD:
"What is next to the pizza boxes?"
"How are the glasses positioned relative to the food?"
"What is between the two glasses?"

These require identifying a relationship between objects.

The same principle applies to every tool:
the query itself, not merely the notes, must demonstrate why the
target tool is appropriate.

- Return exactly ONE query.
- The query must be grounded in this actual image.
- Do not invent visible content.
- Do not force the target tool.
- Reject weak/forced tool-image combinations.
- Do not use technical terminology unless the query structure
  calls for a direct tool request.
- Keep the query natural and realistic.
- Make ambiguity meaningful rather than artificial.
- Objects must actually be visible.
- Notes must be EXACTLY ONE short sentence (max ~12 words).
  Capture only the core reason the tool fits -- no elaboration,
  no restating the query, no re-listing objects.
  Example GOOD: "Multiple faces are clearly visible and locatable."
  Example BAD: "The image clearly shows several people standing
  together in a park setting, with their faces visible enough
  that a face detector could identify and localize each one."
"""

    return prompt

Cell 10 — Gemini call

In [ ]:
# ============================================================================
# Cell 8a -- Gemini judge setup (secrets-based, Kaggle-safe)
# ============================================================================
# ============================================================================
# Cell 8a -- Gemini judge setup (secrets-based, Kaggle-safe)
# ============================================================================

import os
import tempfile
import time

from google import genai
from google.genai import errors as genai_errors


def get_secret(name, required=True):
    try:
        from kaggle_secrets import UserSecretsClient

        val = UserSecretsClient().get_secret(name)

        if val:
            return val

    except Exception:
        pass

    val = os.environ.get(name)

    if val:
        return val

    if required:
        raise RuntimeError(
            f"Secret '{name}' not found. "
            f"Add via Kaggle: Add-ons -> Secrets."
        )

    return None


# ============================================================================
# Secrets
# ============================================================================

GEMINI_API_KEY = get_secret(
    "GEMINI_API_KEY",
    required=False
)

GCP_SERVICE_ACCOUNT = get_secret(
    "GCP_SERVICE_ACCOUNT_JSON",
    required=False
)


# ============================================================================
# CURRENT GOOGLE CLOUD PROJECT
# ============================================================================

GCP_PROJECT = "project-64aa92ad-1c89-485a-b9c"
GCP_LOCATION = "us-central1"

JUDGE_MODEL = "gemini-2.5-flash"


# ============================================================================
# CREATE GEMINI CLIENT
# ============================================================================

if GCP_SERVICE_ACCOUNT:

    # ------------------------------------------------------------
    # Vertex AI
    # ------------------------------------------------------------

    print("Using Vertex AI")
    print(f"Project: {GCP_PROJECT}")
    print(f"Location: {GCP_LOCATION}")
    print(f"Model: {JUDGE_MODEL}")

    from google.oauth2 import service_account

    with tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".json",
        delete=False
    ) as f:
        f.write(GCP_SERVICE_ACCOUNT)
        _sa_path = f.name

    _creds = service_account.Credentials.from_service_account_file(
        _sa_path,
        scopes=[
            "https://www.googleapis.com/auth/cloud-platform"
        ],
    )

    judge_client = genai.Client(
        vertexai=True,
        project=GCP_PROJECT,
        location=GCP_LOCATION,
        credentials=_creds,
    )

    os.remove(_sa_path)


elif GEMINI_API_KEY:

    # ------------------------------------------------------------
    # Gemini API
    # ------------------------------------------------------------

    print("Using Gemini API key")

    judge_client = genai.Client(
        api_key=GEMINI_API_KEY
    )


else:

    raise RuntimeError(
        "Neither GEMINI_API_KEY nor GCP_SERVICE_ACCOUNT_JSON "
        "was found in Kaggle Secrets."
    )


# ============================================================================
# RATE LIMITER (new)
# ============================================================================

_last_call_time = [0.0]

def rate_limit_wait():
    """
    Ensures at least MIN_SECONDS_BETWEEN_CALLS seconds pass between
    any two Gemini calls, across the whole notebook run.
    """
    now = time.time()
    elapsed = now - _last_call_time[0]
    if elapsed < MIN_SECONDS_BETWEEN_CALLS:
        time.sleep(MIN_SECONDS_BETWEEN_CALLS - elapsed)
    _last_call_time[0] = time.time()


# ============================================================================
# SMOKE TEST
# ============================================================================

_smoke = judge_client.models.generate_content(
    model=JUDGE_MODEL,
    contents="Reply with exactly: WORKING"
)

print(
    "Gemini judge smoke test:",
    _smoke.text.strip()
)

In [ ]:
def _extract_text(response):
    text = getattr(response, "text", None)
    if text:
        return text.strip()
    reason = "no text returned"
    try:
        feedback = getattr(response, "prompt_feedback", None)
        if feedback is not None and getattr(feedback, "block_reason", None):
            reason = f"prompt blocked: {feedback.block_reason}"
        elif getattr(response, "candidates", None):
            cand = response.candidates[0]
            reason = f"finish_reason={getattr(cand, 'finish_reason', None)}"
            safety = getattr(cand, "safety_ratings", None)
            if safety:
                reason += f" | safety={safety}"
    except Exception:
        pass
    raise RuntimeError(f"Gemini returned no usable text ({reason})")


def call_gemini_for_query(image_path, tool_name, query_structure):
    prompt = build_generation_prompt(tool_name=tool_name, query_structure=query_structure)
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    mime_type, _ = mimetypes.guess_type(str(image_path))
    if mime_type is None:
        mime_type = "image/jpeg"
    image_part = types.Part.from_bytes(data=image_bytes, mime_type=mime_type)
    rate_limit_wait()
    response = judge_client.models.generate_content(
        model=JUDGE_MODEL,
        contents=[prompt, image_part],
        config=types.GenerateContentConfig(temperature=0.8, response_mime_type="application/json")
    )
    return json.loads(_extract_text(response))


def build_object_inventory_prompt():
    return """
You are inspecting an image for a computer vision dataset.

List every clearly visible object, entity, or distinct item in this image.

Rules:
- Only list objects that are clearly visible.
- Use short plain object names (e.g. "pizza", "table", "person", "car").
- Do NOT describe their state, action, color, position, or relationships.
- Do not invent anything that is not actually visible.
- Return between 3 and 15 objects if possible.

Return ONLY valid JSON in this exact format:
{
  "objects": ["object1", "object2", "object3"]
}
"""


def call_gemini_for_object_inventory(image_path):
    prompt = build_object_inventory_prompt()
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    mime_type, _ = mimetypes.guess_type(str(image_path))
    if mime_type is None:
        mime_type = "image/jpeg"
    image_part = types.Part.from_bytes(data=image_bytes, mime_type=mime_type)
    rate_limit_wait()
    response = judge_client.models.generate_content(
        model=JUDGE_MODEL,
        contents=[prompt, image_part],
        config=types.GenerateContentConfig(temperature=0.3, response_mime_type="application/json")
    )
    data = json.loads(_extract_text(response))
    objects = data.get("objects", [])
    if not isinstance(objects, list):
        return []
    return [str(o).strip() for o in objects if str(o).strip()]

Cell 10b — Master object list per image (new, cached)

In [ ]:
IMAGE_OBJECTS_CACHE = {}
BLOCKED_IMAGES = {}

def get_master_objects(image_path):
    image_id = get_image_id(image_path)
    if image_id in IMAGE_OBJECTS_CACHE:
        return IMAGE_OBJECTS_CACHE[image_id]
    try:
        objects = call_gemini_for_object_inventory(image_path)
    except Exception as e:
        print("    Object inventory error:", repr(e))
        objects = []
        BLOCKED_IMAGES[image_id] = str(e)
    objects_str = ", ".join(objects)
    IMAGE_OBJECTS_CACHE[image_id] = objects_str
    return objects_str

Cell 11 — Test ONE image first

In [ ]:
# ============================================================
# SINGLE IMAGE TEST
# ============================================================

test_image = selected_images[0]

test_tool = choose_target_tool(test_image)
test_structure = choose_target_structure(test_tool)

print("Image:", test_image.name)
print("Target tool:", test_tool)
print("Target structure:", test_structure)

result = call_gemini_for_query(
    test_image,
    test_tool,
    test_structure
)

print("\nGemini result:")
print(json.dumps(result, indent=2))

Cell 12 — Validate Gemini output

In [ ]:
# ============================================================
# VALIDATE GENERATED RESULT
# ============================================================

VALID_AMBIGUITY = {
    "low",
    "medium",
    "high"
}


def validate_result(result, target_tool):

    if not isinstance(result, dict):
        return False, "Result is not a dictionary."

    if result.get("applicable") is not True:
        return False, "Tool not applicable."

    query = result.get("query")

    if not query or not isinstance(query, str):
        return False, "Missing query."

    query = query.strip()

    if len(query) < 5:
        return False, "Query too short."

    ambiguity = result.get("ambiguity_level")

    if ambiguity not in VALID_AMBIGUITY:
        return False, f"Invalid ambiguity: {ambiguity}"

    objects = result.get("objects", [])

    if not isinstance(objects, list):
        return False, "objects must be a list."

    requires_context = result.get(
        "requires_image_context"
    )

    if requires_context not in {"yes", "no"}:
        return False, "Invalid requires_image_context."

    notes = result.get("notes", "")

    if not isinstance(notes, str):
        return False, "notes must be a string."

    return True, "valid"

Cell 13 — Generate one accepted query

In [ ]:
# ============================================================
# GENERATE ONE ACCEPTED QUERY
# ============================================================

def generate_one_query(image_path):

    rejected_attempts = []

    for attempt in range(MAX_RETRIES_PER_SLOT):

        # ----------------------------------------------------
        # Select target tool
        # ----------------------------------------------------

        target_tool = choose_target_tool(image_path)

        # ----------------------------------------------------
        # Select target structure
        # ----------------------------------------------------

        target_structure = choose_target_structure(
            target_tool
        )

        # Count this as an attempt regardless of outcome, so
        # tools that keep getting rejected don't permanently
        # sit at the bottom of the selection pool.
        tool_attempt_counts[target_tool] += 1

        print(
            f"  Attempt {attempt + 1}: "
            f"{target_tool} | {target_structure}"
        )

        try:

            result = call_gemini_for_query(
                image_path,
                target_tool,
                target_structure
            )

        except Exception as e:

            print(
                "    Gemini error:",
                repr(e)
            )

            rejected_attempts.append(
                {
                    "tool": target_tool,
                    "structure": target_structure,
                    "reason": f"API error: {e}"
                }
            )

            backoff = RETRY_BASE_DELAY * (attempt + 1)
            print(f"    Backing off {backoff}s before retrying...")
            time.sleep(backoff)

            continue

        # ----------------------------------------------------
        # Validate
        # ----------------------------------------------------

        valid, reason = validate_result(
            result,
            target_tool
        )

        if not valid:

            print(
                "    Rejected:",
                reason
            )

            rejected_attempts.append(
                {
                    "tool": target_tool,
                    "structure": target_structure,
                    "reason": reason
                }
            )

            continue

        # ----------------------------------------------------
        # ACCEPT
        # ----------------------------------------------------

        print(
            "    ACCEPTED:",
            result["query"]
        )

        return {
            "target_tool": target_tool,
            "target_structure": target_structure,
            "result": result
        }

    return None


# ============================================================
# FALLBACK QUERY GENERATION (new)
# ============================================================
# Only called when an image got ZERO accepted queries through
# the normal balanced tool/structure selection above.
# Tries FALLBACK_TOOLS in order. Still validated via Gemini --
# never force-accepted.

def generate_fallback_query(image_path):

    print("  No query via normal selection -- trying fallback tools...")

    for tool_name in FALLBACK_TOOLS:

        for attempt in range(MAX_RETRIES_PER_SLOT):

            target_structure = choose_target_structure(tool_name)

            print(
                f"  Fallback attempt: "
                f"{tool_name} | {target_structure}"
            )

            try:

                result = call_gemini_for_query(
                    image_path,
                    tool_name,
                    target_structure
                )

            except Exception as e:

                print("    Gemini error:", repr(e))

                backoff = RETRY_BASE_DELAY * (attempt + 1)
                time.sleep(backoff)

                continue

            valid, reason = validate_result(result, tool_name)

            if not valid:
                print("    Rejected:", reason)
                continue

            print("    FALLBACK ACCEPTED:", result["query"])

            return {
                "target_tool": tool_name,
                "target_structure": target_structure,
                "result": result
            }

    print("  Fallback exhausted -- no query for this image.")
    return None

Cell 14 — Convert accepted result into your CSV row

In [ ]:
# ============================================================
# BUILD CSV ROW
# ============================================================

query_counter = 1


def build_csv_row(
    image_path,
    accepted,
    objects_str
):
    global query_counter

    tool_name = accepted["target_tool"]
    result = accepted["result"]

    row = {
        "query_id": f"q{query_counter:05d}",

        "query": result["query"].strip(),

        # IMPORTANT:
        # tool_label in the output CSV comes from tool_id
        # in tool_reference.csv
        "tool_label": TOOL_ID_MAP[tool_name],

        "tool_name": tool_name,

        # Image filename without extension
        # e.g. 12576.jpg -> 12576
        "image_id": get_image_id(image_path),

        # Shared master object list for this image_id --
        # same string for every accepted query on this image.
        "objects": objects_str,

        "ambiguity_level":
            result["ambiguity_level"],

        "requires_image_context":
            result["requires_image_context"],

        # ALWAYS "known"
        "image_pool": "known",

        "notes":
            result.get("notes", "").strip(),

        # Internal column for pilot QA.
        # This will be removed before final CSV export.
        "query_structure":
            accepted["target_structure"]
    }

    query_counter += 1

    return row


Cell 15 — Main 50-image pilot

In [ ]:
# ============================================================
# RUN 50-IMAGE PILOT
# ============================================================

all_rows = []
skipped_images = []
start_time = time.time()

for image_index, image_path in enumerate(
    selected_images,
    start=1
):
    print("\n" + "=" * 80)
    print(
        f"IMAGE {image_index}/{len(selected_images)}: "
        f"{image_path.name}"
    )
    print("=" * 80)

    image_id = get_image_id(image_path)

    # --------------------------------------------------------
    # Master object inventory for this image (1 call, cached)
    # --------------------------------------------------------
    image_objects_str = get_master_objects(image_path)
    print(f"  Master objects: {image_objects_str}")

    # Skip immediately if this image is blocked (e.g. safety
    # filter) -- don't waste retries on every tool.
    if image_id in BLOCKED_IMAGES:
        print(
            f"  SKIPPING image -- Gemini returned no usable "
            f"content for it ({BLOCKED_IMAGES[image_id]})."
        )
        skipped_images.append(image_path.name)
        continue

    image_generated = 0

    for slot in range(QUERIES_PER_IMAGE):
        print(
            f"\n  Query slot "
            f"{slot + 1}/{QUERIES_PER_IMAGE}"
        )
        accepted = generate_one_query(
            image_path
        )
        if accepted is None:
            print(
                "  Could not generate an accepted "
                "query for this slot."
            )
            continue

        row = build_csv_row(
            image_path,
            accepted,
            image_objects_str
        )
        all_rows.append(row)

        # ----------------------------------------------------
        # Update quotas ONLY after acceptance
        # ----------------------------------------------------
        tool_name = accepted["target_tool"]
        structure = accepted["target_structure"]
        tool_counts[tool_name] += 1
        structure_counts[structure] += 1
        tool_structure_counts[
            (tool_name, structure)
        ] += 1
        image_generated += 1
        print(
            f"  Saved candidate #{len(all_rows)}"
        )

    # --------------------------------------------------------
    # FALLBACK: only runs if this image got ZERO queries
    # --------------------------------------------------------
    if image_generated == 0:
        accepted = generate_fallback_query(image_path)
        if accepted is not None:
            row = build_csv_row(
                image_path,
                accepted,
                image_objects_str
            )
            all_rows.append(row)
            tool_name = accepted["target_tool"]
            structure = accepted["target_structure"]
            tool_counts[tool_name] += 1
            structure_counts[structure] += 1
            tool_structure_counts[(tool_name, structure)] += 1
            image_generated += 1
            print(
                f"  Saved fallback candidate #{len(all_rows)}"
            )

    print(
        f"\n  Generated for this image: "
        f"{image_generated}/{QUERIES_PER_IMAGE}"
    )

elapsed = time.time() - start_time

print("\n" + "=" * 80)
print("PILOT COMPLETE")
print("=" * 80)
print("Images:", len(selected_images))
print("Accepted queries:", len(all_rows))
print("Skipped (blocked) images:", len(skipped_images))
if skipped_images:
    print("  ->", skipped_images)
print(f"Time: {elapsed / 60:.2f} minutes")

Cell 16 — Create DataFrame with exact column order

In [ ]:
# ============================================================
# CREATE OUTPUT DATAFRAME
# ============================================================

OUTPUT_COLUMNS = [
    "query_id",
    "query",
    "tool_label",
    "tool_name",
    "image_id",
    "objects",
    "ambiguity_level",
    "requires_image_context",
    "image_pool",
    "notes"
]

results_df = pd.DataFrame(
    all_rows,
    columns=OUTPUT_COLUMNS
)

print(
    f"Generated {len(results_df)} rows."
)

display(results_df.head(10))

Cell 17 — Save to CSV

In [ ]:
# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print(
    f"Saved to:\n{OUTPUT_CSV}"
)

Cell 19 — Check the generated data

In [ ]:
# ============================================================
# BASIC DATA VALIDATION
# ============================================================

print("Number of rows:", len(results_df))

print("\nMissing values:")
print(results_df[OUTPUT_COLUMNS].isnull().sum())

print("\nImage pool values:")
print(results_df["image_pool"].value_counts())

print("\nAmbiguity values:")
print(results_df["ambiguity_level"].value_counts())

print("\nUnique images:")
print(results_df["image_id"].nunique())

print("\nUnique tools used:")
print(results_df["tool_name"].nunique())

Cell 20 — Tool distribution

In [ ]:
# ============================================================
# TOOL DISTRIBUTION
# ============================================================

tool_distribution = (
    results_df
    .groupby(
        ["tool_label", "tool_name"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "tool_label"
    )
)

display(tool_distribution)

In [ ]:
# ============================================================
# TOOLS WITH ZERO QUERIES
# ============================================================

used_tools = set(
    results_df["tool_name"]
)

unused_tools = [
    tool
    for tool in TOOLS
    if tool not in used_tools
]

print(
    f"Tools with at least one query: "
    f"{len(used_tools)}/{len(TOOLS)}"
)

print(
    f"Tools with zero queries: "
    f"{len(unused_tools)}"
)

if unused_tools:
    print("\nUnused tools:")
    for tool in unused_tools:
        print(
            f"{TOOL_ID_MAP[tool]} -> {tool}"
        )

In [ ]:
# ============================================================
# REBUILD RESULTS DATAFRAME INCLUDING INTERNAL STRUCTURE
# ============================================================

results_df = pd.DataFrame(all_rows)

print("Columns currently available:")
print(results_df.columns.tolist())

if "query_structure" not in results_df.columns:
    raise ValueError(
        "query_structure is missing from all_rows. "
        "Make sure you replaced Cell 14 with the corrected "
        "build_csv_row() function before running Cell 15."
    )

print("\nquery_structure successfully found.")
print(
    "Number of generated queries:",
    len(results_df)
)

Cell 21 — Query structure distribution

In [ ]:
# ============================================================
# QUERY STRUCTURE DISTRIBUTION
# ============================================================

if "query_structure" in results_df.columns:

    structure_distribution = (
        results_df["query_structure"]
        .value_counts()
        .reindex(
            STRUCTURE_NAMES,
            fill_value=0
        )
    )

    print(structure_distribution)

else:

    print(
        "query_structure is not present. "
        "Add it as an internal column in Cell 14 "
        "to evaluate the five query structures."
    )

In [ ]:
FINAL_COLUMNS = [
    "query_id",
    "query",
    "query_structure",
    "tool_label",
    "tool_name",
    "image_id",
    "objects",
    "ambiguity_level",
    "requires_image_context",
    "image_pool",
    "notes"
]

results_df = pd.DataFrame(all_rows)

# Make sure all required columns exist
missing_columns = set(FINAL_COLUMNS) - set(results_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing columns in results_df: {missing_columns}"
    )

# Keep only the final columns, in the desired order
results_df = results_df[FINAL_COLUMNS]

print("Final columns:")
print(results_df.columns.tolist())

print("\nNumber of queries:", len(results_df))

display(results_df.head())

Cell 22 — Tool × Query Structure

In [ ]:
# ============================================================
# TOOL × QUERY STRUCTURE
# ============================================================

if "query_structure" in results_df.columns:

    tool_structure_table = pd.crosstab(
        results_df["tool_name"],
        results_df["query_structure"]
    )

    # Make sure all five structures appear as columns
    tool_structure_table = (
        tool_structure_table
        .reindex(
            columns=STRUCTURE_NAMES,
            fill_value=0
        )
    )

    display(tool_structure_table)

else:

    print(
        "query_structure is not available."
    )

Cell 23 — Ambiguity distribution

In [ ]:
# ============================================================
# AMBIGUITY DISTRIBUTION
# ============================================================

ambiguity_distribution = (
    results_df["ambiguity_level"]
    .value_counts()
    .reindex(
        ["low", "medium", "high"],
        fill_value=0
    )
)

display(
    ambiguity_distribution
)

Cell 24 — Inspect all generated queries

In [ ]:
# ============================================================
# REVIEW GENERATED QUERIES
# ============================================================

review_columns = [
    "query_id",
    "query",
    "tool_label",
    "tool_name",
    "image_id",
    "ambiguity_level",
    "notes"
]

if "query_structure" in results_df.columns:
    review_columns.append("query_structure")

display(
    results_df[review_columns]
    .sort_values("query_id")
    .reset_index(drop=True)
)

Cell 25 — Check duplicate queries

In [ ]:
# ============================================================
# DUPLICATE QUERY CHECK
# ============================================================

normalized_queries = (
    results_df["query"]
    .astype(str)
    .str.lower()
    .str.strip()
)

duplicates = results_df[
    normalized_queries.duplicated(
        keep=False
    )
].copy()

print(
    "Duplicate query count:",
    len(duplicates)
)

if len(duplicates) > 0:

    display(
        duplicates[
            [
                "query_id",
                "query",
                "tool_name",
                "image_id"
            ]
        ]
        .sort_values("query")
    )
else:

    print("No duplicate queries found.")

Cell 26 — Final CSV

In [ ]:
# ============================================================
# SAVE FINAL CSV
# ============================================================

FINAL_COLUMNS = [
    "query_id",
    "query",
    "tool_label",
    "tool_name",
    "image_id",
    "objects",
    "ambiguity_level",
    "requires_image_context",
    "image_pool",
    "notes"
]

final_df = results_df[
    FINAL_COLUMNS
].copy()

# Enforce image_pool
final_df["image_pool"] = "known"

final_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("=" * 70)
print("FINAL CSV SAVED")
print("=" * 70)

print("Path:", OUTPUT_CSV)
print("Rows:", len(final_df))
print("Columns:", len(final_df.columns))

display(final_df.head(10))

Cell 27 — Final sanity check

In [ ]:
# ============================================================
# FINAL SANITY CHECK
# ============================================================

assert list(final_df.columns) == [
    "query_id",
    "query",
    "tool_label",
    "tool_name",
    "image_id",
    "objects",
    "ambiguity_level",
    "requires_image_context",
    "image_pool",
    "notes"
]

assert (final_df["image_pool"] == "known").all()

assert final_df["query"].notna().all()

assert final_df["tool_name"].isin(TOOLS).all()

assert final_df["ambiguity_level"].isin(
    ["low", "medium", "high"]
).all()

assert final_df["requires_image_context"].isin(
    ["yes", "no"]
).all()

assert final_df["query_id"].is_unique

print("✅ ALL FINAL CHECKS PASSED")
print()
print(f"Queries: {len(final_df)}")
print(f"Images: {final_df['image_id'].nunique()}")
print(f"Tools represented: {final_df['tool_name'].nunique()}/{len(TOOLS)}")
print("image_pool: known")

Cell 26 — Export CSV

In [ ]:
OUTPUT_CSV = "/kaggle/working/human_queries_pilot_50images.csv"

results_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print(f"CSV saved to: {OUTPUT_CSV}")
print(f"Rows: {len(results_df)}")
print(f"Columns: {len(results_df.columns)}")